# Ornstein–Uhlenbeck (mean reverting)

Mean-reverting log price: `dx = kappa (theta - x) dt + sigma dW`, with `kappa = ln 2 / half_life`, simulated with the exact discretisation. `THETA` is the mean *log* price.

Fantasma simulates **log returns** (log price differences): one row per step, labelled at the end of the step. `fm.to_prices`, `fm.to_log_prices` and `fm.to_returns` turn them into other quantities.

Edit the parameters cell and re-run all: the same parameters and seed always give the same paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import fantasma as fm

## Parameters

Model parameters are annualised; `dt` is the step length over a 365.25-day year.

In [ ]:
SEED = 42
N_PATHS = 5
START = "2024-01-01"
FREQ = "1d"  # Almanac frequency ("1m", "10m", "1h", "1d") or any pandas offset
N_STEPS = 365 * 5

THETA = np.log(100.0)  # mean log price
HALF_LIFE = 0.1  # years
SIGMA = 0.3
X0 = np.log(120.0)  # starting log price (None starts at THETA)
ASSET = "OU"

## Simulate

In [ ]:
model = fm.OrnsteinUhlenbeck(theta=THETA, half_life=HALF_LIFE, sigma=SIGMA, asset=ASSET, x0=X0)
model

In [ ]:
paths = fm.simulate(
    model,
    start=START,
    freq=FREQ,
    n_steps=N_STEPS,
    seed=SEED,
    n_paths=N_PATHS,
    return_hidden=True,
)
log_returns, hidden = paths[0]  # log returns; hidden state on the full grid
log_returns.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for i, (path_log_returns, _) in enumerate(paths):
    path_prices = fm.to_prices(path_log_returns)  # start included, from the recorded start price
    for asset in path_prices.columns:
        ax.plot(path_prices.index, path_prices[asset], lw=0.8, label=f"path {i} {asset}")
ax.set_title("Ornstein–Uhlenbeck (mean reverting)")
ax.set_ylabel("price")
if len(paths) * paths[0][0].shape[1] <= 10:
    ax.legend(fontsize=8)
plt.show()

In [ ]:
prices = fm.to_prices(log_returns)
ax = plt.figure(figsize=(10, 2.5)).gca()
ax.axhline(np.exp(THETA), color="k", lw=1, ls="--", label="exp(theta)")
ax.plot(prices.index, prices[ASSET], lw=0.8, label="path 0")
ax.legend()
plt.show()

## Check: AR(1) regression

Regressing `x[t+1]` on `x[t]` gives slope `b = exp(-kappa dt)` and intercept `a = theta (1 - b)`.

In [ ]:
dt = fm.RegularGrid(START, FREQ, n_steps=1).dt
x = fm.to_log_prices(log_returns)[ASSET].to_numpy()
b, a = np.polyfit(x[:-1], x[1:], 1)
print(f"half-life = {HALF_LIFE:.4f} y, estimate = {-np.log(2) * dt / np.log(b):.4f} y")
print(f"theta     = {THETA:.4f},   estimate = {a / (1 - b):.4f}")